In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import time

CATALOG = "fahrlytics"
SILVER  = f"{CATALOG}.silver"
GOLD    = f"{CATALOG}.gold"

spark.sql(f"USE CATALOG {CATALOG}")

print(f"Catalog           : {CATALOG}")
print(f"Target Gold Table : {GOLD}.payment_features")

In [0]:
print("Reading clean Silver payments, trips, and customers...")

df_payments = spark.table(f"{SILVER}.payments")
df_trips    = spark.table(f"{SILVER}.trips")
df_cust     = spark.table(f"{SILVER}.customers")

In [0]:
print("Constructing ground-truth fraud label...")

df_labeled_payments = (
    df_payments
    .withColumn(
        "is_fraud_label",
        F.when(
            (F.col("is_shared_card_fraud") == True) |
            (F.col("is_orphan_payment") == True) |
            (F.col("is_fare_mismatch") == True),
            1
        ).otherwise(0)
    )
)

In [0]:
print("Computing card syndication and transaction velocity signals...")

# Historical distinct customers per card
card_window = Window.partitionBy("card_hash")
# Rolling customer payment sequence
cust_time_window = Window.partitionBy("customer_id").orderBy("payment_timestamp")

df_velocity_features = (
    df_labeled_payments
    # Distinct accounts sharing the same payment card
    .withColumn("card_linked_accounts_count",
        F.when(F.col("card_hash").isNotNull(),
               F.size(F.collect_set("customer_id").over(card_window)))
        .otherwise(1)
    )
    # Total volume run on this card
    .withColumn("card_lifetime_tx_count",
        F.when(F.col("card_hash").isNotNull(),
               F.count("payment_id").over(card_window))
        .otherwise(1)
    )
    # Elapsed seconds since previous transaction by same customer
    .withColumn("prev_payment_timestamp", F.lag("payment_timestamp", 1).over(cust_time_window))
    .withColumn("seconds_since_last_payment",
        F.when(F.col("prev_payment_timestamp").isNotNull(),
               F.unix_timestamp("payment_timestamp") - F.unix_timestamp("prev_payment_timestamp"))
        .otherwise(86400) # Default to 24 hours if first transaction
    )
    .drop("prev_payment_timestamp")
)

In [0]:
print("Joining trip and customer context to compute deviation metrics...")

# Select minimal trip attributes
trips_context = df_trips.select(
    F.col("trip_id").alias("t_trip_id"),
    F.col("distance_km").alias("trip_distance_km"),
    F.col("trip_duration_min"),
    F.col("vehicle_type"),
    F.col("surge_multiplier"),
    F.col("is_ghost_trip")
)

# Select minimal customer attributes
cust_context = df_cust.select(
    F.col("customer_id").alias("c_customer_id"),
    F.col("signup_date"),
    F.col("rating").alias("customer_rating"),
    F.col("device_type")
)

df_enriched = (
    df_velocity_features
    .join(trips_context, df_velocity_features.trip_id == trips_context.t_trip_id, how="left")
    .join(cust_context, df_velocity_features.customer_id == cust_context.c_customer_id, how="left")
    .drop("t_trip_id", "c_customer_id")
)

In [0]:
print("Calculating financial heuristic ratios...")

# Compute benchmark city median fare to detect artificial price inflation
city_median_window = Window.partitionBy("city")

df_features = (
    df_enriched
    # 1. Fare per kilometer ratio
    .withColumn("amount_per_km",
        F.when(F.col("trip_distance_km") > 0,
               F.round(F.col("amount") / F.col("trip_distance_km"), 2))
        .otherwise(F.col("amount"))
    )
    # 2. Account age in days at the exact time of payment
    .withColumn("account_age_days_at_tx",
        F.greatest(F.lit(0), F.datediff(F.to_date("payment_timestamp"), F.col("signup_date")))
    )
    # 3. Transaction hour & Late-night anomaly flag (00:00 - 05:00)
    .withColumn("tx_hour", F.hour("payment_timestamp"))
    .withColumn("is_late_night_tx", F.col("tx_hour").isin(0, 1, 2, 3, 4))
    # 4. Round amount indicator (fraud testing frequently uses integer amounts like €50.00, €100.00)
    .withColumn("is_round_amount", (F.col("amount") % 10.0 == 0.0) | (F.col("amount") % 5.0 == 0.0))
    # 5. Method indicators
    .withColumn("is_credit_card", F.col("payment_method") == "Credit Card")
    .withColumn("is_paypal", F.col("payment_method") == "PayPal")
    # 6. City median deviation
    .withColumn("city_avg_amount", F.round(F.avg("amount").over(city_median_window), 2))
    .withColumn("amount_to_city_avg_ratio", F.round(F.col("amount") / F.greatest(F.col("city_avg_amount"), F.lit(1.0)), 2))
    # Fill defaults for orphan or missing trip records
    .na.fill({
        "trip_distance_km": 0.0,
        "trip_duration_min": 0.0,
        "vehicle_type": "Standard",
        "surge_multiplier": 1.0,
        "is_ghost_trip": False,
        "customer_rating": 4.5,
        "account_age_days_at_tx": 0,
        "device_type": "iOS"
    })
    # Drop raw leakage columns so downstream ML notebooks never access them
    .drop(
        "is_fraud_card",
        "is_orphan_payment",
        "is_fare_mismatch",
        "is_shared_card_fraud",
        "fare_difference",
        "is_customer_mismatched",
        "card_hash"
    )
    .withColumn("_feature_calculated_at", F.current_timestamp())
)

In [0]:
print(f"Writing feature table to {GOLD}.payment_features...")
t0 = time.time()

(
    df_features.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("is_fraud_label")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD}.payment_features")
)

print(f"Gold payment features written in {round(time.time() - t0, 2)}s.")

print("Running OPTIMIZE with Z-ORDER on customer_id and payment_timestamp...")
spark.sql(f"OPTIMIZE {GOLD}.payment_features ZORDER BY (customer_id, payment_timestamp)")
print("OPTIMIZE completed.")

In [0]:
print("=== FAHRLYTICS GOLD PAYMENT FEATURES AUDIT ===\n")

df_gold_payments = spark.table(f"{GOLD}.payment_features")
total_records = df_gold_payments.count()
print(f"Total transactions in feature store: {total_records:,}\n")

print("Target Class Distribution (is_fraud_label):")
df_gold_payments.groupBy("is_fraud_label").count().withColumn(
    "percentage", F.round(F.col("count") / total_records * 100, 3)
).show()

print("Feature Distribution by Fraud Class:")
df_gold_payments.groupBy("is_fraud_label").agg(
    F.round(F.avg("amount"), 2).alias("avg_amount"),
    F.round(F.avg("card_linked_accounts_count"), 2).alias("avg_card_sharing"),
    F.round(F.avg("amount_per_km"), 2).alias("avg_amount_per_km"),
    F.round(F.avg("account_age_days_at_tx"), 1).alias("avg_account_age_days"),
    F.round(F.avg(F.col("is_late_night_tx").cast("int")), 3).alias("late_night_rate")
).show()

print("Feature Matrix Sample:")
df_gold_payments.select(
    "payment_id", "city", "amount", "payment_method",
    "card_linked_accounts_count", "seconds_since_last_payment",
    "amount_to_city_avg_ratio", "is_fraud_label"
).show(5, truncate=False)